In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import cv2
import random
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
# Load the label file
label_file_path = './Data_Label.txt'
label_df = pd.read_csv(label_file_path, sep=" ", header=None, names=["Filename", "Pitch", "Roll", "Depth"])

# label_df["Normalized_Depth"] = label_df.groupby("Pitch")["Depth"].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0.0)

def normalize_depth(group):
    min_depth = group["Depth"].min()
    max_depth = group["Depth"].max()
    if min_depth == max_depth:
        group["Normalized_Depth"] = 0.0  # 避免除以0
    else:
        group["Normalized_Depth"] = (group["Depth"] - min_depth) / (max_depth - min_depth)
    return group

label_df = label_df.groupby(["Pitch", "Roll"], group_keys=False).apply(normalize_depth)

print("Data Structure(First 4 image):")
print(label_df.head())

unique_pitch = label_df['Pitch'].unique()
unique_roll = label_df['Roll'].unique()

print("\nPitch Values contain:", unique_pitch)
print("Roll Values contain:", unique_roll)
print("\nStatistics of Depth Values:")
print(label_df['Normalized_Depth'].describe())
print("\nTotal number of images:", len(label_df))


In [ ]:
def load_images(image_dir, filenames):
    """
    Load images based on the provided filenames.

    Parameters:
        image_dir (str): Directory containing the images.
        filenames (list): List of image filenames.

    Returns:
        list: Loaded images with their file names and labels.
    """
    images = []
    for filename in filenames:
        img_path = os.path.join(image_dir, filename)
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            labels = label_df[label_df['Filename'] == filename].iloc[0]
            images.append((filename, img, labels))
        else:
          # may be there is missing value? no idea just in case
            images.append((filename, None, None))
    return images

image_directory = './Data_Images/20-Robot_with_holder'

In [ ]:
def display_images(images):
    num_samples = len(images)
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))

    for ax, (filename, img, labels) in zip(axes, images):
        if img is not None:
            ax.imshow(img)
            ax.set_title(f"{filename}", fontsize=10)
            ax.set_xlabel(f"Pitch: {labels['Pitch']}\nRoll: {labels['Roll']}\nDepth: {labels['Depth']:.2f}", fontsize=8)
            ax.axis('off')
        else:
            ax.set_title(f"{filename}\nNot Found", fontsize=10)
            ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# random display 6 images to show sucessful loading
sample_filenames = label_df.sample(n=6, random_state=42)['Filename'].tolist()
# sample_filenames
loaded_images = load_images(image_directory, sample_filenames)
display_images(loaded_images)


In [ ]:
#Show the distribution of pitch and roll in dataset
pitch_counts = label_df['Pitch'].value_counts()
roll_counts = label_df['Roll'].value_counts()

pitch_counts.plot(kind='bar', title='Pitch Distribution')
plt.show()

roll_counts.plot(kind='bar', title='Roll Distribution')
plt.show()

In [ ]:
# # Create a combined Pose column for easier analysis
# label_df['Pose'] = list(zip(label_df['Pitch'],label_df['Roll']))

# # Count the occurrences of each unique pose
# pose_counts = label_df['Pose'].value_counts()
# print("Most Frequent Poses:\n", pose_counts.head())
# print("\nLeast Frequent Poses:\n", pose_counts.tail())

In [ ]:
# # Boxplot of Depth values grouped by Pose
# label_df.boxplot(column='Normalized_Depth', by='Pose', figsize=(15, 6), vert=False)
# plt.title('Depth Distribution by Pose')
# plt.xlabel('Depth')
# plt.ylabel('Pose (Pitch, Roll)')
# plt.show()

In [ ]:
def normalize_images(images):
    normalized_images = []
    for filename, img, labels in images:
        if img is not None:
            img = img / 255.0  # Normalize pixel values to [0, 1]
        normalized_images.append((filename, img, labels))
    return normalized_images

# def standardize_depth_values(df):
#     mean_depth = df['Depth'].mean()
#     std_depth = df['Depth'].std()
#     df['Depth'] = (df['Depth'] - mean_depth) / std_depth
#     print("\nDepth values standardized.")
#     return df
# image_directory = './Data_Images/20-Robot_with_holder'
# all_images = load_images(image_directory, label_df['Filename'].tolist())
# all_images

In [ ]:
def split_group(group, train_ratio=2/3, val_ratio=1/6, test_ratio=1/6):
    if len(group) < 4:
        return group, pd.DataFrame(), pd.DataFrame()

    train_size = int(len(group) * train_ratio)
    val_size = int(len(group) * val_ratio)

    train_group, temp_group = train_test_split(group, train_size=train_size, random_state=42)
    val_group, test_group = train_test_split(temp_group, train_size=val_size, random_state=42)

    return train_group, val_group, test_group

train_list, val_list, test_list = [], [], []

for _, group in label_df.groupby(["Pitch", "Roll"]):
    train_group, val_group, test_group = split_group(group)
    train_list.append(train_group)
    val_list.append(val_group)
    test_list.append(test_group)

train_files = pd.concat(train_list).reset_index(drop=True)
val_files = pd.concat(val_list).reset_index(drop=True)
test_files = pd.concat(test_list).reset_index(drop=True)

# len(train_files), len(val_files), len(test_files)
print("\nDataset Splitting Complete:")
print(f"Train set size: {len(train_files)}")
print(f"Validation set size: {len(val_files)}")
print(f"Test set size: {len(test_files)}")

In [ ]:
output_directory = './Data_Spilt'
os.makedirs(output_directory, exist_ok=True)
train_files.to_csv(os.path.join(output_directory, "train_labels.txt"), sep=" ", index=False, header=False)
val_files.to_csv(os.path.join(output_directory, "val_labels.txt"), sep=" ", index=False, header=False)
test_files.to_csv(os.path.join(output_directory, "test_labels.txt"), sep=" ", index=False, header=False)

print(f"数据划分完成：Train({len(train_files)}), Val({len(val_files)}), Test({len(test_files)})")

def save_split_images(data_split, image_dir, output_dir, split_name):
    split_dir = os.path.join(output_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    for filename in data_split['Filename']:
        src_path = os.path.join(image_dir, filename)
        dst_path = os.path.join(split_dir, filename)
        if os.path.exists(src_path):
            shutil.copy(src_path, dst_path)
        else:
            print(f"Warning: File {src_path} not found!")

    print(f"Images for {split_name} split saved to: {split_dir}")

save_split_images(train_files, image_directory, output_directory, "train")
save_split_images(val_files, image_directory, output_directory, "val")
save_split_images(test_files, image_directory, output_directory, "test")

---

## Approach 1: Combinational model

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F
import pandas as pd
import os
import numpy as np

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, num_classes_pitch, num_classes_roll, backbone_name):
        super(MultiTaskModel, self).__init__()

        if backbone_name == "resnet18":
            self.backbone = models.resnet18(pretrained=True)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()  # Remove the fully connected layer
        elif backbone_name == "resnet50":
            self.backbone = models.resnet50(pretrained=True)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif backbone_name == "vgg16":
            self.backbone = models.vgg16(pretrained=True)
            num_features = self.backbone.classifier[-1].in_features
            self.backbone.classifier = nn.Sequential(*list(self.backbone.classifier.children())[:-1])

            # Fine-tune VGG16: Freeze early layers
            for param in self.backbone.features[:10].parameters():
                param.requires_grad = False
            # Keep later layers trainable
            for param in self.backbone.features[10:].parameters():
                param.requires_grad = True
        elif backbone_name == "vit":
            self.backbone = models.vit_b_16(pretrained=True)
            num_features = self.backbone.heads.head.in_features
            self.backbone.heads.head = nn.Identity()
        else:
            raise ValueError("Unsupported backbone model. Choose from 'resnet18', 'resnet50', 'vgg16', or 'vit'.")

        # Pitch classifier head
        self.pitch_head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes_pitch)
        )

        # Roll classifier head
        self.roll_head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes_roll)
        )

        # Depth regression head
        self.depth_head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        if hasattr(self.backbone, 'heads'):  # ViT-specific forward
            features = self.backbone(x)
        else:  # ResNet and VGG forward
            features = self.backbone(x)

        pitch_output = self.pitch_head(features)
        roll_output = self.roll_head(features)
        depth_output = self.depth_head(features)

        return pitch_output, roll_output, depth_output


In [ ]:
# Define loss functions
classification_loss_fn = nn.CrossEntropyLoss()  # For Pitch and Roll
regression_loss_fn = nn.MSELoss()  # For Depth
#For special use in calculating the Vit regression loss
# regression_loss_fn = nn.SmoothL1Loss()
def multi_task_loss(pitch_pred, pitch_target, roll_pred, roll_target, depth_pred, depth_target, alpha=1.0, beta=1.0):
    pitch_loss = classification_loss_fn(pitch_pred, pitch_target)
    roll_loss = classification_loss_fn(roll_pred, roll_target)
    depth_loss = regression_loss_fn(depth_pred, depth_target)

    total_loss = alpha * (pitch_loss + roll_loss) + beta * depth_loss
    return total_loss


In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        images, pitch_targets, roll_targets, depth_targets = batch

        # Move data to the device
        images = images.to(device)
        pitch_targets = pitch_targets.to(device)
        roll_targets = roll_targets.to(device)
        depth_targets = depth_targets.to(device)

        # Forward pass
        pitch_pred, roll_pred, depth_pred = model(images)

        # Compute loss
        loss = multi_task_loss(pitch_pred, pitch_targets, roll_pred, roll_targets, depth_pred, depth_targets)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    return total_loss / len(dataloader)


## Dataloader and Model training

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

In [ ]:
data_dir = "./Data_Video/20-Robot_with_holder"
pitch_poses = []
roll_poses = []
for image_name in os.listdir(data_dir):
    parts = image_name.split("_")
    pitch = int(parts[0][1:])  # Extract number after 'P'
    roll = int(parts[1][1:])   # Extract number after 'R'
    # print(pitch, roll)
    pitch_poses.append(pitch)
    roll_poses.append(roll)

unique_pitches = list(set(pitch_poses))
unique_rolls = list(set(roll_poses))
unique_pitches.sort()
unique_rolls.sort()

pitch_to_class = {pitch_angle: idx for idx, pitch_angle in enumerate(unique_pitches)}
roll_to_class = {roll_angle: idx for idx, roll_angle in enumerate(unique_rolls)}
# len(pitch_to_class), len(roll_to_class)

# # Define the model
# num_classes_pitch = len(pitch_to_class)
# num_classes_roll = len(roll_to_class)
# model = MultiTaskModel(backbone_name='resnet50', num_classes_pitch=num_classes_pitch, num_classes_roll=num_classes_roll)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class MultiTaskDataset(Dataset):
    def __init__(self, txt_file, image_dir, transform=None):
        self.data = pd.read_csv(txt_file, sep=" ", header=None)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Get image file path
        img_name = os.path.join(self.image_dir, self.data.iloc[idx, 0])
        
        if not os.path.exists(img_name):
            print(f"Warning: Image {img_name} not found!")
            
        # Load image
        image = Image.open(img_name).convert("RGB")

        # Get labels
        pitch = self.data.iloc[idx, 1]
        roll = self.data.iloc[idx, 2]
        norm_depth = self.data.iloc[idx, 4]

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        # Convert labels to tensors
        pitch = pitch_to_class[pitch]
        roll = roll_to_class[roll]
        norm_depth = torch.tensor(norm_depth, dtype=torch.float32)  # Regression label

        return image, pitch, roll, norm_depth

# Define transformations
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224
    transforms.ToTensor(),          # Convert images to PyTorch tensors
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize using ImageNet stats
])

# File paths
train_txt = './Data_Spilt/train_labels.txt'
val_txt = './Data_Spilt/val_labels.txt'
test_txt = './Data_Spilt/test_labels.txt'

# Correct image directories for each split
train_image_dir = './Data_Spilt/train'
val_image_dir = './Data_Spilt/val'
test_image_dir = './Data_Spilt/test'

# Create datasets
train_dataset = MultiTaskDataset(txt_file=train_txt, image_dir=train_image_dir, transform=data_transforms)
val_dataset = MultiTaskDataset(txt_file=val_txt, image_dir=val_image_dir, transform=data_transforms)
test_dataset = MultiTaskDataset(txt_file=test_txt, image_dir=test_image_dir, transform=data_transforms)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, scheduler, device, num_epochs=10):
    model.to(device)

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0

        for images, pitch_targets, roll_targets, depth_targets in train_loader:
            # Move data to device
            images = images.to(device, dtype=torch.float32)
            pitch_targets = pitch_targets.to(device, dtype=torch.long)
            roll_targets = roll_targets.to(device, dtype=torch.long)
            depth_targets = depth_targets.to(device, dtype=torch.float32).view(-1, 1)

            # Forward pass
            pitch_pred, roll_pred, depth_pred = model(images)

            # Compute loss
            print("pitch: ", pitch_pred, pitch_targets, "roll: ", roll_pred, roll_targets)
            loss = multi_task_loss(pitch_pred, pitch_targets, roll_pred, roll_targets, depth_pred, depth_targets)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Learning rate scheduler step
        scheduler.step()

        # Validation phase
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for images, pitch_targets, roll_targets, depth_targets in val_loader:
                # Move data to device
                images = images.to(device, dtype=torch.float32)
                pitch_targets = pitch_targets.to(device, dtype=torch.long)
                roll_targets = roll_targets.to(device, dtype=torch.long)
                depth_targets = depth_targets.to(device, dtype=torch.float32).view(-1, 1)

                # Forward pass
                pitch_pred, roll_pred, depth_pred = model(images)

                # Compute loss
                loss = multi_task_loss(pitch_pred, pitch_targets, roll_pred, roll_targets, depth_pred, depth_targets)

                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")


In [ ]:
import matplotlib.pyplot as plt
def train_model_with_optimization(
    model, train_loader, val_loader, optimizer, scheduler, device,
    num_epochs, alpha, beta, dropout_rate):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    criterion_classification = torch.nn.CrossEntropyLoss()
    criterion_regression = torch.nn.MSELoss()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss = 0.0
        correct_train_classification = 0
        total_train_samples = 0

        for batch in train_loader:
            inputs, labels_pitch, labels_roll, depths = batch
            inputs = inputs.to(device)
            labels_pitch = labels_pitch.to(device)
            labels_roll = labels_roll.to(device)
            depths = depths.to(device)

            optimizer.zero_grad()

            outputs_pitch, outputs_roll, outputs_depth = model(inputs)

            # print("pitch: ", outputs_pitch, labels_pitch, "roll: ", outputs_roll, labels_roll)

            # Calculate losses
            loss_pitch = criterion_classification(outputs_pitch, labels_pitch)
            loss_roll = criterion_classification(outputs_roll, labels_roll)
            loss_depth = criterion_regression(outputs_depth, depths)
            loss = alpha * (loss_pitch + loss_roll) + beta * loss_depth

            loss.backward()
            optimizer.step()

            # Track loss and accuracy
            train_loss += loss.item() * inputs.size(0)
            _, predicted_pitch = outputs_pitch.max(1)
            _, predicted_roll = outputs_roll.max(1)
            correct_train_classification += (
                (predicted_pitch == labels_pitch).sum().item() +
                (predicted_roll == labels_roll).sum().item()
            )
            total_train_samples += 2 * inputs.size(0)  # pitch + roll

        train_losses.append(train_loss / len(train_loader.dataset))
        train_accuracies.append(correct_train_classification / total_train_samples)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val_classification = 0
        total_val_samples = 0

        with torch.no_grad():
            for batch in val_loader:
                inputs, labels_pitch, labels_roll, depths = batch
                inputs = inputs.to(device)
                labels_pitch = labels_pitch.to(device)
                labels_roll = labels_roll.to(device)
                depths = depths.to(device)

                outputs_pitch, outputs_roll, outputs_depth = model(inputs)

                # Calculate losses
                loss_pitch = criterion_classification(outputs_pitch, labels_pitch)
                loss_roll = criterion_classification(outputs_roll, labels_roll)
                loss_depth = criterion_regression(outputs_depth, depths)
                loss = alpha * (loss_pitch + loss_roll) + beta * loss_depth

                val_loss += loss.item() * inputs.size(0)
                _, predicted_pitch = outputs_pitch.max(1)
                _, predicted_roll = outputs_roll.max(1)
                correct_val_classification += (
                    (predicted_pitch == labels_pitch).sum().item() +
                    (predicted_roll == labels_roll).sum().item()
                )
                total_val_samples += 2 * inputs.size(0)

        val_losses.append(val_loss / len(val_loader.dataset))
        val_accuracies.append(correct_val_classification / total_val_samples)

        scheduler.step()

        print(
            f"Epoch {epoch}/{num_epochs}, "
            f"Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, "
            f"Train Acc: {train_accuracies[-1]:.4f}, Val Acc: {val_accuracies[-1]:.4f}"
        )

    # Plot training process
    plot_training_process(train_losses, val_losses, train_accuracies, val_accuracies)

    return model


def plot_training_process(train_losses, val_losses, train_accuracies, val_accuracies):
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(14, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label="Train Loss")
    plt.plot(epochs, val_losses, label="Val Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label="Train Accuracy")
    plt.plot(epochs, val_accuracies, label="Val Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# device = torch.device('cuda')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = MultiTaskModel(backbone_name='resnet50', num_classes_pitch=len(pitch_to_class), num_classes_roll=len(roll_to_class)).to(device)

num_epochs = 20
alpha = 1.0
beta = 0.8
learning_rate = 0.001
dropout_rate = 0.5

# Optimizer and scheduler
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

train_model_with_optimization(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    device,
    num_epochs=num_epochs,
    alpha=alpha,
    beta=beta,
    dropout_rate=dropout_rate
)

In [ ]:
def save_model(model: torch.nn.Module, path: str) -> None:
    torch.save(model.state_dict(), path)
    print(f"Model saved to {path}")

def load_model(model: torch.nn.Module, path: str, device: str) -> torch.nn.Module:
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    print(f"Model loaded from {path}")
    return model

In [ ]:
model_save_path = "/content/drive/MyDrive/DL_PartA/Model/"
# Save the model
# save_model(model, model_save_path + "NewNew_ResNet5005.pth")

## Evaluate the model on test set and see the result

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, precision_score, recall_score, f1_score, accuracy_score

In [ ]:
def evaluate_model(model: torch.nn.Module, test_loader: torch.utils.data.DataLoader, device: str) -> tuple[dict, dict]:
    """
    Evaluate the model on the test set and compute metrics.

    Parameters:
        model (torch.nn.Module): Trained model.
        test_loader (torch.utils.data.DataLoader): Test data loader.
        device (str): Device to evaluate on ('cuda' or 'cpu').

    Returns:
        tuple[dict, dict]:
            - Dictionary containing metrics for pitch, roll, and depth.
            - Dictionary containing predictions and targets for pitch, roll, and depth.
    """
    model.eval()
    model.to(device)

    # Initialize accumulators
    all_pitch_preds = []
    all_pitch_targets = []
    all_roll_preds = []
    all_roll_targets = []
    all_depth_preds = []
    all_depth_targets = []

    with torch.no_grad():
        for images, pitch_targets, roll_targets, depth_targets in test_loader:
            # Move data to device
            images = images.to(device, dtype=torch.float32)
            pitch_targets = pitch_targets.to(device, dtype=torch.long)
            roll_targets = roll_targets.to(device, dtype=torch.long)
            depth_targets = depth_targets.to(device, dtype=torch.float32).view(-1, 1)

            # Forward pass
            pitch_preds, roll_preds, depth_preds = model(images)

            # Store predictions and targets
            all_pitch_preds.extend(torch.argmax(pitch_preds, dim=1).cpu().numpy())
            all_pitch_targets.extend(pitch_targets.cpu().numpy())

            all_roll_preds.extend(torch.argmax(roll_preds, dim=1).cpu().numpy())
            all_roll_targets.extend(roll_targets.cpu().numpy())

            all_depth_preds.extend(depth_preds.cpu().numpy())
            all_depth_targets.extend(depth_targets.cpu().numpy())

    # Compute metrics for pitch classification
    pitch_accuracy = accuracy_score(all_pitch_targets, all_pitch_preds)
    pitch_precision = precision_score(all_pitch_targets, all_pitch_preds, average="weighted")
    pitch_recall = recall_score(all_pitch_targets, all_pitch_preds, average="weighted")
    pitch_f1 = f1_score(all_pitch_targets, all_pitch_preds, average="weighted")

    # Compute metrics for roll classification
    roll_accuracy = accuracy_score(all_roll_targets, all_roll_preds)
    roll_precision = precision_score(all_roll_targets, all_roll_preds, average="weighted")
    roll_recall = recall_score(all_roll_targets, all_roll_preds, average="weighted")
    roll_f1 = f1_score(all_roll_targets, all_roll_preds, average="weighted")

    # Compute RMSE for depth regression
    depth_rmse = np.sqrt(mean_squared_error(all_depth_targets, all_depth_preds))

    metrics = {
        "Pitch Accuracy": pitch_accuracy,
        "Pitch Precision": pitch_precision,
        "Pitch Recall": pitch_recall,
        "Pitch F1 Score": pitch_f1,
        "Roll Accuracy": roll_accuracy,
        "Roll Precision": roll_precision,
        "Roll Recall": roll_recall,
        "Roll F1 Score": roll_f1,
        "Depth RMSE": depth_rmse,
    }

    return metrics, {
        "pitch_preds": all_pitch_preds,
        "pitch_targets": all_pitch_targets,
        "roll_preds": all_roll_preds,
        "roll_targets": all_roll_targets,
        "depth_preds": all_depth_preds,
        "depth_targets": all_depth_targets
    }

In [ ]:
def plot_results(metrics: dict, data: dict) -> None:
    """
    Visualize the evaluation results.

    Parameters:
        metrics (dict): Evaluation metrics.
        data (dict): Contains all predictions and targets.
    """
    print("Evaluation Metrics:")
    for metric, value in metrics.items():
        print(f"{metric:<20}: {value:.4f}")

    # Confusion Matrix for Pitch
    cm_pitch = confusion_matrix(data["pitch_targets"], data["pitch_preds"])
    disp_pitch = ConfusionMatrixDisplay(confusion_matrix=cm_pitch, display_labels=[f"Pitch {i}" for i in range(cm_pitch.shape[0])])
    disp_pitch.plot(cmap=plt.cm.Blues)
    plt.title("Pitch Confusion Matrix")
    plt.show()

    # Confusion Matrix for Roll
    cm_roll = confusion_matrix(data["roll_targets"], data["roll_preds"])
    disp_roll = ConfusionMatrixDisplay(confusion_matrix=cm_roll, display_labels=[f"Roll {i}" for i in range(cm_roll.shape[0])])
    disp_roll.plot(cmap=plt.cm.Blues)
    plt.title("Roll Confusion Matrix")
    plt.show()

    # Scatter plot for Depth
    plt.scatter(data["depth_targets"], data["depth_preds"], alpha=0.5, label="Predictions")
    plt.plot([min(data["depth_targets"]), max(data["depth_targets"])], [min(data["depth_targets"]), max(data["depth_targets"])], 'r--', label="Ideal Fit")
    plt.xlabel("True Depth")
    plt.ylabel("Predicted Depth")
    plt.title("Depth Regression: True vs Predicted")
    plt.legend(loc="upper left")
    plt.text(0.70, 0.10, f"RMSE: {metrics['Depth RMSE']:.4f}", transform=plt.gca().transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.5))
    plt.show()


In [ ]:
model = MultiTaskModel(
    num_classes_pitch=71,
    num_classes_roll=71,
    backbone_name="resnet50"
).to(device)

# Load model state dict with strict=False to handle size mismatches
def load_model_with_flexibility(model, path, device):
    model.load_state_dict(torch.load(path, map_location=device), strict=True)
    model.eval()  # Set the model to evaluation mode
    return model

model = load_model_with_flexibility(model, model_save_path + "New2_ResNet50.pth", device)
metrics, data = evaluate_model(model, test_loader, device)
plot_results(metrics, data)

---

## Model Ramk and the Winner in different task

In [ ]:
def evaluate_model_performance(model, test_loader, device):
    model.eval()
    all_pitch_preds, all_pitch_targets = [], []
    all_roll_preds, all_roll_targets = [], []
    all_depth_preds, all_depth_targets = [], []

    with torch.no_grad():
        for batch in test_loader:
            images, pitch_targets, roll_targets, depth_targets = batch
            images = images.to(device)
            pitch_targets = pitch_targets.to(device)
            roll_targets = roll_targets.to(device)
            depth_targets = depth_targets.to(device)

            # Forward pass
            pitch_preds, roll_preds, depth_preds = model(images)

            # Store results
            all_pitch_preds.extend(torch.argmax(pitch_preds, dim=1).cpu().numpy())
            all_pitch_targets.extend(pitch_targets.cpu().numpy())
            all_roll_preds.extend(torch.argmax(roll_preds, dim=1).cpu().numpy())
            all_roll_targets.extend(roll_targets.cpu().numpy())
            all_depth_preds.extend(depth_preds.squeeze().cpu().numpy())
            all_depth_targets.extend(depth_targets.cpu().numpy())

    # Classification Metrics
    pitch_accuracy = accuracy_score(all_pitch_targets, all_pitch_preds)
    pitch_f1 = f1_score(all_pitch_targets, all_pitch_preds, average="weighted")
    roll_accuracy = accuracy_score(all_roll_targets, all_roll_preds)
    roll_f1 = f1_score(all_roll_targets, all_roll_preds, average="weighted")

    # Regression Metric
    depth_rmse = mean_squared_error(all_depth_targets, all_depth_preds, squared=False)

    return {
        "Pitch Accuracy": pitch_accuracy,
        "Pitch F1": pitch_f1,
        "Roll Accuracy": roll_accuracy,
        "Roll F1": roll_f1,
        "Depth RMSE": depth_rmse
    }


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

def evaluate_Ultra_model_performance(model, test_loader, device):
    model.eval()
    all_pitch_preds, all_pitch_targets = [], []
    all_roll_preds, all_roll_targets = [], []
    all_depth_preds, all_depth_targets = [], []

    with torch.no_grad():
        for batch in test_loader:
            images, pitch_targets, roll_targets, depth_targets = batch
            images = images.to(device)
            pitch_targets = pitch_targets.to(device)
            roll_targets = roll_targets.to(device)
            depth_targets = depth_targets.to(device)

            # Forward pass
            pitch_preds, roll_preds, depth_preds = model(images)

            # Store results
            all_pitch_preds.extend(torch.argmax(pitch_preds, dim=1).cpu().numpy())
            all_pitch_targets.extend(pitch_targets.cpu().numpy())
            all_roll_preds.extend(torch.argmax(roll_preds, dim=1).cpu().numpy())
            all_roll_targets.extend(roll_targets.cpu().numpy())
            all_depth_preds.extend(depth_preds.squeeze().cpu().numpy())
            all_depth_targets.extend(depth_targets.cpu().numpy())

    # Classification Metrics
    pitch_accuracy = accuracy_score(all_pitch_targets, all_pitch_preds)
    pitch_f1 = f1_score(all_pitch_targets, all_pitch_preds, average="weighted")
    roll_accuracy = accuracy_score(all_roll_targets, all_roll_preds)
    roll_f1 = f1_score(all_roll_targets, all_roll_preds, average="weighted")

    # Regression Metric
    depth_rmse = np.sqrt(mean_squared_error(all_depth_targets, all_depth_preds))
    return {
        "Pitch Accuracy": pitch_accuracy,
        "Pitch F1": pitch_f1,
        "Roll Accuracy": roll_accuracy,
        "Roll F1": roll_f1,
        "Depth RMSE": depth_rmse
    }


In [ ]:
def compare_models(models, test_loader, device, classification_weight=0.7, regression_weight=0.3):
    results = []

    for model_name, model in models.items():
        print(f"Evaluating {model_name}...")
        metrics = evaluate_Ultra_model_performance(model, test_loader, device)

        # Normalize RMSE (lower is better, so invert it)
        normalized_depth_rmse = 1 / (1 + metrics["Depth RMSE"])

        # Calculate combined score
        combined_score = (
            classification_weight * (metrics["Pitch Accuracy"] + metrics["Roll Accuracy"]) / 2 +
            regression_weight * normalized_depth_rmse
        )

        results.append({
            "Model": model_name,
            "Metrics": metrics,
            "Combined Score": combined_score
        })

    # Sort results by combined score
    results = sorted(results, key=lambda x: x["Combined Score"], reverse=True)

    return results


In [ ]:
# Define your models with paths
model_paths = {
    "ResNet18": "/content/drive/MyDrive/DL_PartA/Model/New1_ResNet18.pth",
    "ResNet50": "/content/drive/MyDrive/DL_PartA/Model/New_ResNet50.pth",
    "ResNet50 with Regularization": "/content/drive/MyDrive/DL_PartA/Model/New2_ResNet50.pth",
    "test ResNet50 model": "/content/drive/MyDrive/DL_PartA/Model/Last_New3_ResNet.pth.pth",
    "VGG16": "/content/drive/MyDrive/DL_PartA/Model/Improve_VGG16.pth",
    "VGG16 Imporve":"/content/drive/MyDrive/DL_PartA/Model/Imporve2_VGG16.pth",
    "VGG16 with more epoch": "/content/drive/MyDrive/DL_PartA/Model/Improve3_Early_VGG16.pth",
    "ViT": "/content/drive/MyDrive/DL_PartA/Model/VIT.pth",
}

# Initialize and load models
models_database = {}
for model_name, model_path in model_paths.items():
    try:
        if "ResNet18" in model_name:
            model_eva = MultiTaskModel(num_classes_pitch=71, num_classes_roll=71, backbone_name="resnet18").to(device)
        elif "ResNet50" in model_name:
            model_eva = MultiTaskModel(num_classes_pitch=71, num_classes_roll=71, backbone_name="resnet50").to(device)
        elif "VGG16" in model_name:
            model_eva = MultiTaskModel(num_classes_pitch=71, num_classes_roll=71, backbone_name="vgg16").to(device)
        elif "ViT" in model_name:
            model_eva = MultiTaskModel(num_classes_pitch=71, num_classes_roll=71, backbone_name="vit").to(device)
        else:
            raise ValueError(f"Unknown model name: {model_name}")


        # Load the saved weights into the model
        state_dict = torch.load(model_path, map_location=device)
        model_eva.load_state_dict(state_dict, strict=False)
        model_eva.eval()  # Set the model to evaluation mode

        models_database[model_name] = model_eva
        print(f"Successfully loaded weights for {model_name}")

    except Exception as e:
        print(f"Error loading model {model_name}: {e}")

for model_name in models_database:
    print(f"Model: {model_name}, Device: {next(models_database[model_name].parameters()).device}")

# Evaluate and rank models
print("\nEvaluating and ranking models...")
results = compare_models(models_database, test_loader, device)

# Display results
print("\nEvaluation Results:")
for idx, result in enumerate(results, start=1):
    print(f"Rank {idx}: {result['Model']}")
    for metric, value in result["Metrics"].items():
        print(f"    {metric}: {value:.4f}")
    print(f"    Combined Score: {result['Combined Score']:.4f}\n")


# The code below are use to debug, or check information from model or data tensor

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)  # Example input tensor
pitch_pred, roll_pred, depth_pred = model(dummy_input)
print("Pitch Output Shape:", pitch_pred.shape)
print("Roll Output Shape:", roll_pred.shape)
print("Depth Output Shape:", depth_pred.shape)

In [ ]:
for batch in train_loader:
    images, pitch_targets, roll_targets, depth_targets = batch
    print("Images Shape:", images.shape)
    print("Pitch Targets:", pitch_targets)
    print("Roll Targets:", roll_targets)
    print("Depth Targets:", depth_targets)
    break

In [ ]:
def check_target_distribution(data_loader, dataset_name):
    """
    Checks the size and range of target data in the dataset.

    Parameters:
        data_loader (DataLoader): DataLoader for the dataset (train/val/test).
        dataset_name (str): Name of the dataset (e.g., "train", "val", "test").
    """
    pitch_values = []
    roll_values = []
    depth_values = []

    for images, pitch_targets, roll_targets, depth_targets in data_loader:
        pitch_values.extend(pitch_targets.cpu().numpy())
        roll_values.extend(roll_targets.cpu().numpy())
        depth_values.extend(depth_targets.cpu().numpy().flatten())

    print(f"--- {dataset_name} Dataset ---")
    print(f"Number of Samples: {len(pitch_values)}")
    print(f"Pitch Targets - Min: {min(pitch_values)}, Max: {max(pitch_values)}")
    print(f"Roll Targets - Min: {min(roll_values)}, Max: {max(roll_values)}")
    print(f"Depth Targets - Min: {min(depth_values):.4f}, Max: {max(depth_values):.4f}")
    print(f"Depth Targets Mean: {np.mean(depth_values):.4f}, Std Dev: {np.std(depth_values):.4f}")
    print("\n")

# Run the function for train, val, and test loaders
check_target_distribution(train_loader, "Train")
check_target_distribution(val_loader, "Validation")
check_target_distribution(test_loader, "Test")